In [0]:
%pip install langchain==0.1.14 typing_extensions==4.8.0 --force-reinstall
dbutils.library.restartPython()


In [0]:
%pip install -U langchain langchain-community


In [0]:
%pip install mlflow[databricks] --upgrade
dbutils.library.restartPython()

In [0]:
%pip install langchain-databricks
dbutils.library.restartPython()


In [0]:
import pandas as pd
import io
from langchain_databricks import ChatDatabricks

# Step 1: Load Unity Catalog table
df = spark.table("solutions_catalog.pih_schema.consideration_data")

# Step 2: Convert to Pandas CSV
# Step 2: Convert to Pandas CSV with pipe delimiter
pdf = df.toPandas()
csv_buffer = io.StringIO()
pdf.to_csv(csv_buffer, index=False, sep='|')  # Set delimiter to pipe
csv_content = csv_buffer.getvalue()


# Step 3: Prepare Prompt
dataset_knowledge = """
This dataset contains campaign performance data across multiple dimensions.
- `Study_ID`: Unique identifier for the study.
- `Brand_Name`, `Advertiser Name`: Brand and advertiser involved in the campaign.
- `Campaign_ID`, `Campaign Name`: Campaign identifiers and descriptive names.
- `Publisher_Group`: The publisher segment or channel type.
- `Sampled_Exposures`: Number of impressions considered.
- `Engagement_Rate`: Engagement metric used to evaluate success.
- `brand_index`, `linear_index`, `digital_index`: Effectiveness benchmarks across brand, linear, and digital channels.
"""

prompt = f"""
You are a campaign performance analyst.

Given this sample data:
{csv_content}

Dataset Knowledge:
{dataset_knowledge}

Instructions:
- Identify the campaigns with the highest and lowest Engagement Rate (%).
- Highlight any unusual or extreme values in brand_index, linear_index, or digital_index.
- Provide insight into how different Publisher Groups are performing.
- Suggest areas for optimization based on the observed metrics.
- Do not generate code blocks.
"""


# Step 4: Run Databricks-hosted LLM
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=2048)
response = llm.invoke(prompt)

# Step 5: Display result
print("Insight from LLaMA 3.3:")
print(response.content.strip())


Problem	How RAG Solves It

Too much data to fit in prompt	- Only the most relevant rows are retrieved

Need contextual insights	- Retrieved rows provide grounding for the LLM

Questions vary by user intent	- LLM can reason flexibly with retrieved info

Slow full scan with LLM	- Vector DB allows fast retrieval + summaries

In [0]:
prompt = f"""

You are a campaign performance analyst.

Dataset knowledge:
{dataset_knowledge}

Sample data:
{csv_content}

Instructions:
- Filter the data to include only rows where Study_ID = 8031961277. Remove all the duplicate rows.
- Always exclude rows where campaign name is empty or null. Exclude all rows where campaign id is 0 or null.
- Summarize the following:
  - List all campaigns as list_of_campaigns based on campaign name. 
  - Count the number of unique campaigns based on unique campaign name as number_of_campaigns. 
  - List all unique publisher groups as list_of_publisher_groups based on publisher group. 
  - Average engagement rate as avg_engagement_rate also include all the values for Study ID. 
  - List all engagement rate as list_of_engagement_rate for all unique campaigns. Include all the values for Study ID. 
  - Average brand index as avg_brand_index for all unique campaigns. 
  - Average linear index as avg_linear_index for all unique campaigns. 
  - Average digital index as avg_digital_index for all unique campaigns.
- Round all averages to 2 decimal places.
- Do not generate any code block as output.
- Do not provide any explaination


Output format:
1. Study ID: 8031961277
1. Number of Unique campaigns : <number_of_campaigns>
2. List of campaigns:  comma separated list_of_campaigns
3. Publisher Group : comma separated list_of_publisher_groups
4. List of engagement rate: comma separated list_of_engagement_rate
5. Average engagement rate : avg_engagement_rate
6. Average brand index : avg_brand_index
7. Average linear index : avg_linear_index
8. Average digital index : avg_digital_index

"""


# Step 4: Run Databricks-hosted LLM
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=2048)
response = llm.invoke(prompt)

# Step 5: Display result
print("Insight from LLaMA 3.3:")
print(response.content.strip())